# 📊 Deep Research — Batch Report Evaluation for the domain `Ecology`

This notebook evaluates **multiple** Deep Research `.md` reports using **YESciEval rubrics**.

### Steps
1. **Read** all `.md` report files in a given folder
2. For each report, **read the research question** from the domain CSV 
3. **Run the YESciEval judge** across all rubrics for every report
4. **Aggregate** rubric scores → category scores (0–1) per report
5. **Accumulate** scores across reports → config-level mean ± std
6. Display a **per-report score table** and a **6-panel batch quality plot**
7. Save:
   - `batch_reports.csv` — one row per report
   - `batch_config_summary.csv` — one row per config with mean/std
   - `batch_quality_dimensions.png` — 6-panel plot
   - `<stem>_scores.json` — full rubric detail per report

---
**Prerequisites:** `pip install yescieval python-dotenv matplotlib numpy`

## 1 — Imports & Setup

In [ ]:
!pip install yescieval python-dotenv matplotlib -q

In [ ]:
import re, json, csv
from pathlib import Path
from typing import Dict, Tuple, List
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from dotenv import load_dotenv
load_dotenv()
print("✅ Core imports loaded")

In [ ]:
from yescieval import CustomAutoJudge, VocabularyInjector, ExampleInjector
from yescieval.rubric.pointwise.depth      import TemporalPrecision, CausalReasoning, MechanisticUnderstanding
from yescieval.rubric.pointwise.breadth    import ContextCoverage, ScopeCoverage, DimensionCoverage, ScaleCoverage, MethodCoverage
from yescieval.rubric.pointwise.rigor      import EpistemicCalibration, ExplicitUncertainty, QuantitativeEvidenceAndUncertainty
from yescieval.rubric.pointwise.innovation import StateOfTheArtAndNovelty
from yescieval.rubric.pointwise.gap        import GapIdentification
print("✅ YESciEval imports loaded")

## 2 — Configuration

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
REPORTS_DIR   = "your_report_path_here.md" 
QUESTIONS_CSV = "your_questions_csv_path_here.csv" 
OUTPUT_DIR  = "your_output_dir_here" 
DOMAIN         = "ecology" # nlp or ecology
MODEL_ID       = "your_model_id_here"
DEVICE         = "cpu"    # set to "cuda" if GPU is available
HF_TOKEN       = "your_hf_token_here"      # optional HuggingFace token
MAX_NEW_TOKENS = 2048
REPORT_GLOB    = "*.md"
ALL_CONFIGS = ['d1_b1', 'd1_b4', 'd4_b1', 'd4_b4']

# Category -> list of (RubricClass, weight) tuples
CATEGORIES: Dict[str, List[Tuple[type, float]]] = {
    "depth": [
        (TemporalPrecision,         1/3),
        (CausalReasoning,           1/3),
        (MechanisticUnderstanding,  1/3),
    ],
    "breadth": [
        (ContextCoverage,   1/5),
        (ScopeCoverage,     1/5),
        (DimensionCoverage, 1/5),
        (MethodCoverage,    1/5),
        (ScaleCoverage,     1/5),
    ],
    "rigor": [
        (EpistemicCalibration,               1/3),
        (ExplicitUncertainty,                1/3),
        (QuantitativeEvidenceAndUncertainty, 1/3),
    ],
    "innovation": [
        (StateOfTheArtAndNovelty, 1.0),
    ],
    "gap": [
        (GapIdentification, 1.0),
    ],
}

CAT_NAMES  = list(CATEGORIES.keys())          # ['depth','breadth','rigor','innovation','gap']
SCORE_COLS = CAT_NAMES + ['overall']          # category scores + overall

print(f"Reports dir : {REPORTS_DIR}")
print(f"Questions   : {QUESTIONS_CSV}")
print(f"Model       : {MODEL_ID}")
print(f"Categories  : {CAT_NAMES}")
print(f"All configs : {ALL_CONFIGS}")

## 3 — Helper Functions

In [ ]:
def parse_config(stem: str) -> str:
    """Extracts dX_bY from filename, e.g. '1_o3-mini_orkg_d1_b1' -> 'd1_b1'."""
    m = re.search(r'd(\d+)_b(\d+)', stem)
    return f"d{m.group(1)}_b{m.group(2)}" if m else 'unknown'


def parse_report_number(stem: str) -> int:
    """Extracts the leading integer from a filename, e.g. '1_o3-mini' -> 1."""
    m = re.match(r'(\d+)[_\-]', stem)
    return int(m.group(1)) if m else -1


def load_question_from_csv(csv_path: str, report_number: int) -> str:
    """Reads the research question at 1-based row position from the CSV."""
    for encoding in ['cp1252', 'utf-8', 'utf-8-sig', 'latin-1']:
        try:
            with open(csv_path, newline='', encoding=encoding) as f:
                reader = csv.DictReader(f)
                for row_num, row in enumerate(reader, start=1):
                    if row_num == report_number:
                        q = row.get('Your research question.', '').strip()
                        return q if q else f'Empty question at row {report_number}'
            return f'Row {report_number} not found in CSV'
        except UnicodeDecodeError:
            continue
    return 'Could not open CSV with any known encoding'


def parse_judge_result(result, rubric_name: str) -> Tuple[int, str]:
    """Extracts (rating 1-5, rationale) from a YESciEval judge result."""
    if isinstance(result, str):
        cleaned = re.sub(r'<think>.*?</think>', '', result, flags=re.DOTALL).strip()
        json_str, depth, start = None, 0, None
        for i, ch in enumerate(cleaned):
            if ch == '{':
                if depth == 0: start = i
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0 and start is not None:
                    json_str = cleaned[start:i+1]; break
        if json_str:
            try: result = json.loads(json_str)
            except json.JSONDecodeError: pass
        if isinstance(result, str):
            try: return int(result.strip()), ''
            except ValueError: return 0, result

    if isinstance(result, dict):
        inner = result.get(rubric_name) or result.get(rubric_name.lower())
        if inner is None and len(result) == 1:
            inner = next(iter(result.values()))
        if isinstance(inner, dict):
            return int(inner.get('rating', 0)), str(inner.get('rationale', ''))
        if 'rating' in result:
            return int(result['rating']), str(result.get('rationale', ''))
        return 0, str(result)

    if hasattr(result, 'rating') and hasattr(result, 'rationale'):
        return int(result.rating), str(result.rationale)
    if hasattr(result, 'score'):
        try: return int(round(float(result.score))), ''
        except (ValueError, TypeError): pass
    try: return int(str(result).strip()), ''
    except ValueError: pass
    return 0, str(result)


def compute_category_score(rubric_scores: List[Tuple[float, float]]) -> float:
    """
    Weighted mean of (score, weight) pairs, normalized from 1-5 to 0-1.
    Formula: (weighted_mean - 1) / (5 - 1)
    """
    total_w = sum(w for _, w in rubric_scores)
    if not total_w:
        return 0.0
    weighted_mean = sum(s * w for s, w in rubric_scores) / total_w
    return round((weighted_mean - 1) / 4, 4)


def discover_reports(reports_dir: str, glob_pattern: str) -> List[Path]:
    """Returns matching files sorted by their leading report number."""
    return sorted(
        Path(reports_dir).glob(glob_pattern),
        key=lambda p: parse_report_number(p.stem)
    )


print("✅ Helpers defined")

## 4 — Discover Reports

In [ ]:
report_paths = discover_reports(REPORTS_DIR, REPORT_GLOB)

if not report_paths:
    raise FileNotFoundError(
        f"No files matching '{REPORT_GLOB}' found in: {REPORTS_DIR}\n"
        "Check REPORTS_DIR and REPORT_GLOB in the configuration cell."
    )

print(f"Found {len(report_paths)} report(s) in '{REPORTS_DIR}':")
for i, p in enumerate(report_paths, 1):
    cfg  = parse_config(p.stem)
    rnum = parse_report_number(p.stem)
    print(f"  [{i:>3}] {p.name}   config={cfg}  report#={rnum}")

## 5 — Initialise the YESciEval Judge

In [ ]:
print(f"Loading {MODEL_ID} on {DEVICE} ...")
judge = CustomAutoJudge()
judge.from_pretrained(model_id=MODEL_ID, device=DEVICE, token=HF_TOKEN or None)
print("Judge ready")

## 6 — Batch Evaluation Loop

Each report is scored across all rubrics. Results accumulate in `all_results`.

In [ ]:
all_results: List[Dict] = []
papers = {} 

for report_idx, report_path in enumerate(report_paths, 1):
    stem       = report_path.stem
    config_str = parse_config(stem)
    report_num = parse_report_number(stem)

    print(f"\n{'='*64}")
    print(f"[{report_idx}/{len(report_paths)}]  {report_path.name}")
    print(f"   config={config_str}  report#={report_num}")

    # ── Load content ──────────────────────────────────────────────────────────
    report_md = report_path.read_text(encoding='utf-8', errors='ignore')
    question  = load_question_from_csv(QUESTIONS_CSV, report_num)
    print(f"   question: {question[:110]}{'...' if len(question) > 110 else ''}")

    # ── Score all categories ──────────────────────────────────────────────────
    rubric_raw:      Dict[str, dict]  = {}
    category_scores: Dict[str, float] = {}

    for cat_name, rubric_list in CATEGORIES.items():
        print(f"\n   Category: {cat_name.upper()}")
        cat_rubric_scores: List[Tuple[float, float]] = []

        for RubricClass, weight in rubric_list:
            rname = RubricClass.__name__
            print(f"      [{rname}] w={weight:.3f} ...", end=' ', flush=True)
            rubric = RubricClass(
                papers=papers, question=question,
                answer=report_md, domain=DOMAIN,
                vocabulary=VocabularyInjector(), example=ExampleInjector(),
            )
            raw               = judge.judge(rubric=rubric, max_new_tokens=MAX_NEW_TOKENS)
            rating, rationale = parse_judge_result(raw, rname)
            rubric_raw[rname] = {'rating': rating, 'rationale': rationale}
            print(f"rating={rating}/5")
            cat_rubric_scores.append((float(rating), weight))

        cat_score = compute_category_score(cat_rubric_scores)
        category_scores[cat_name] = cat_score
        print(f"   {cat_name} score = {cat_score:.4f}")

    overall = round(sum(category_scores.values()) / len(category_scores), 4)
    print(f"\n   Overall = {overall:.4f}")

    # Store flat row matching batch_reports.csv column order
    all_results.append({
        'report_stem':   stem,
        'config':        config_str,
        'depth':         category_scores.get('depth', 0.0),
        'breadth':       category_scores.get('breadth', 0.0),
        'rigor':         category_scores.get('rigor', 0.0),
        'innovation':    category_scores.get('innovation', 0.0),
        'gap':           category_scores.get('gap', 0.0),
        'overall':       overall,
        'report_path':   str(report_path),
        # extra detail (used for per-report JSON only)
        'report_number': report_num,
        'domain':        DOMAIN,
        'question':      question,
        'model':         MODEL_ID,
        'rubric_scores': rubric_raw,
    })

print(f"\n{'='*64}")
print(f"Batch complete — {len(all_results)} report(s) evaluated.")

## 7 — Score Accumulation

Group scores by config → compute **mean ± std** per category.

In [ ]:
# Group per-report scores by config
config_buckets: Dict[str, Dict[str, List[float]]] = defaultdict(lambda: defaultdict(list))
for r in all_results:
    for col in SCORE_COLS:
        config_buckets[r['config']][col].append(r[col])

config_summary_rows = []
for cfg in ALL_CONFIGS:
    bucket = config_buckets.get(cfg, {})
    n = len(bucket.get('overall', []))
    row = {'config': cfg, 'n_docs': n}
    for col in SCORE_COLS:
        vals = bucket.get(col, [])
        row[f'{col}_mean'] = round(float(np.mean(vals)), 4) if vals else 0.0
        row[f'{col}_std']  = round(float(np.std(vals)),  4) if len(vals) > 1 else 0.0
    config_summary_rows.append(row)

# ── Display per-report table ──────────────────────────────────────────────────
display(Markdown('### Per-Report Scores'))
rpt_cols = ['report_stem', 'config'] + SCORE_COLS
cw = 24
print(' | '.join(h.ljust(cw) for h in rpt_cols))
print('-' * (cw * len(rpt_cols) + 3 * (len(rpt_cols)-1)))
for r in all_results:
    print(' | '.join(str(r.get(h, '')).ljust(cw) for h in rpt_cols))

# ── Display config summary table ──────────────────────────────────────────────
print()
display(Markdown('### Config Summary (mean ± std)'))
sum_cols = ['config', 'n_docs'] + [f'{c}_mean' for c in SCORE_COLS] + [f'{c}_std' for c in SCORE_COLS]
sw = 16
print(' | '.join(h.ljust(sw) for h in sum_cols))
print('-' * (sw * len(sum_cols) + 3 * (len(sum_cols)-1)))
for row in config_summary_rows:
    print(' | '.join(str(row.get(h, '')).ljust(sw) for h in sum_cols))

## 8 — Batch Quality Plot

6-panel bar chart:
- X-axis: all 4 configs (`d1_b1`, `d1_b4`, `d4_b1`, `d4_b4`) 
- Bar height: config-mean score (0–1) with value label on top

In [ ]:
PANEL_TITLES = {
    'depth':      'Research Depth Score',
    'breadth':    'Research Breadth Score',
    'rigor':      'Scientific Rigor Score',
    'innovation': 'Innovation Score',
    'gap':        'Research Gap Score',
    'overall':    'Overall Quality Score',
}

# Build quick lookup: config -> {panel_key: (mean, std, n)}
cfg_stats: Dict[str, Dict] = {}
for row in config_summary_rows:
    cfg = row['config']
    cfg_stats[cfg] = {
        panel_key: (row.get(f'{panel_key}_mean', 0.0),
                    row.get(f'{panel_key}_std',  0.0),
                    row['n_docs'])
        for panel_key in PANEL_TITLES
    }

xs = list(range(len(ALL_CONFIGS)))

plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for i, (cat, panel_title) in enumerate(PANEL_TITLES.items()):
    ax = axes[i]

    means = [cfg_stats[cfg][cat][0] for cfg in ALL_CONFIGS]
    stds  = [cfg_stats[cfg][cat][1] for cfg in ALL_CONFIGS]
    ns    = [cfg_stats[cfg][cat][2] for cfg in ALL_CONFIGS]

    # Show error bars only when n > 1 (i.e. real variance exists)
    yerr = [s if n > 1 else 0 for s, n in zip(stds, ns)]

    ax.bar(
        xs, means, color='#2980b9', width=0.5,
        yerr=yerr, capsize=4,
        error_kw={'elinewidth': 1.2, 'ecolor': 'black'},
    )
    ax.set_title(panel_title, fontsize=12)
    ax.set_xticks(xs)
    ax.set_xticklabels(ALL_CONFIGS, fontsize=10)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score (0..1)', fontsize=9)

    for x, v in zip(xs, means):
        ax.text(x, v + 0.02, f'{v:.2f}', ha='center', va='bottom', fontsize=9)

n_total = len(all_results)
fig.suptitle(
    f'Research Quality Dimensions Analysis — {DOMAIN} / {DOMAIN} (n={n_total})',
    fontsize=15
)
fig.tight_layout(rect=[0, 0.02, 1, 0.95])
plt.show()
print("Batch plot rendered")

## 9 — Save Outputs to Disk

| File | Contents |
|---|---|
| `batch_reports.csv` | One row per report — stem, config, category scores, overall, path |
| `batch_config_summary.csv` | One row per config — n_docs, mean & std per category |
| `batch_quality_dimensions.png` | 6-panel quality plot |
| `<stem>_scores.json` | Full rubric detail for each report |

In [ ]:
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# ── batch_reports.csv ────────────────────────────────────────────────────────
report_csv_path = out_dir / 'batch_reports.csv'
report_csv_cols = ['report_stem', 'config'] + SCORE_COLS + ['report_path']
with report_csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=report_csv_cols, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(all_results)
print(f"batch_reports.csv        -> {report_csv_path}")

# ── batch_config_summary.csv ─────────────────────────────────────────────────
summary_csv_path = out_dir / 'batch_config_summary.csv'
summary_cols = (
    ['config', 'n_docs']
    + [f'{c}_mean' for c in SCORE_COLS]
    + [f'{c}_std'  for c in SCORE_COLS]
)
with summary_csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=summary_cols, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(config_summary_rows)
print(f"batch_config_summary.csv -> {summary_csv_path}")

# ── Batch plot PNG ────────────────────────────────────────────────────────────
fig_path = out_dir / 'batch_quality_dimensions.png'
fig.savefig(fig_path, dpi=200, bbox_inches='tight')
print(f"batch_quality_dimensions.png -> {fig_path}")

# ── Per-report JSON ───────────────────────────────────────────────────────────
for r in all_results:
    json_path = out_dir / f"{r['report_stem']}_scores.json"
    json_path.write_text(
        json.dumps(r, indent=2, ensure_ascii=False), encoding='utf-8'
    )
    print(f"{json_path.name}")

print("\nAll outputs saved.")